# SciPy Optimizers: Find the Best Answer

**Student:** Syed Tehmeed Jafar  |  **Date:** 25 Nov 2026

**Source:** W3Schools SciPy Optimizers page, with extra explanation and real-life examples.

**Goal:** Understand what an optimizer is, use `root()` and `minimize()` with confidence, and connect it to Machine Learning.

**How to use:** Read a note, run the code cell under it, change one number, and see what changes. At the end, write the idea in your own words.

## Contents

1. What is an optimizer? (the big idea)
2. Setup
3. Job 1: Root finding with `root()`
4. How to read the result object
5. Why the starting guess `x0` matters
6. Job 2: Finding the minimum with `minimize()`
7. See it: a picture of the minimum
8. The `method` argument
9. Local minimum trap
10. `options` and `callback`
11. Trick: how to find a maximum
12. Real-life example: best advertising spend
13. Link to Machine Learning: fit a line by minimizing error
14. Common mistakes
15. Cheat sheet
16. Interview questions
17. Practice exercises
18. My own notes and review plan

## 1. What is an optimizer? (the big idea)

**Short answer:** An optimizer is a tool that finds the **best answer** to a problem by trying values again and again, getting closer each time.

In SciPy, the page says an optimizer does one of two jobs:

| Job | Simple question | SciPy function |
|---|---|---|
| Root finding | At which `x` does the function become **zero**? | `scipy.optimize.root` |
| Minimizing | At which `x` is the function **lowest**? | `scipy.optimize.minimize` |

**Real-life example (walking in fog):** You are on a hill in thick fog. You want the lowest valley, but you cannot see it. You can only feel the slope under your feet. So you take a step downhill, feel the slope again, and repeat until the ground is flat. That is exactly how an optimizer works: it uses small steps and the slope.

**Why this matters for you:** A Machine Learning model makes mistakes (errors). Training the model means changing its numbers until the error is as **small** as possible. So `minimize()` is the idea hiding under linear regression, logistic regression, and even deep learning. The W3Schools page says the same: ML algorithms are complex equations that must be minimized using data.

**Connect to what you know:**

- NumPy gives you arrays and fast maths.
- Pandas gives you tables.
- `scipy.optimize` gives you the "search for the best value" machine that works on top of them.

## 2. Setup

Import only what you need. We import the two functions, plus NumPy and Matplotlib (which you already know).

In [ ]:
import numpy as np                                   # arrays and maths (cos, sum, ...)
import matplotlib.pyplot as plt                      # for the picture later
from scipy.optimize import root, minimize            # the two optimizer functions

## 3. Job 1: Root finding with `root()`

**Short answer:** A **root** is the value of `x` that makes the function equal to **0**. `root()` finds it for you.

**Real-life example (break-even point):** You sell a product for 50 PKR. Each unit costs 30 PKR to make, and you also pay a fixed cost of 2000 PKR. How many units do you need to sell to make **zero profit** (no loss, no gain)?

- Revenue = 50 x units
- Cost = 2000 + 30 x units
- Profit(units) = 50 x units - (2000 + 30 x units) = 20 x units - 2000

Break-even is where **profit = 0**. That is a root problem.

**By hand:** 20 x units - 2000 = 0, so units = 100.

**Why not just use NumPy?** NumPy can solve polynomials (`np.roots`) and straight-line equations (`np.linalg.solve`). But the page's example, `x + cos(x)`, mixes `x` with a cosine. That is a **non-linear** equation, and there is no simple formula. SciPy's `root()` solves it by searching.

**How `root()` works in simple words:** Like adjusting a tap until the water is warm. You start with a guess (`x0`), check how far the answer is from zero, move a little in the right direction, and repeat until it is close enough.

**The two required arguments:**

| Argument | Meaning |
|---|---|
| `fun` | A Python function that returns the value of the equation |
| `x0` | Your first guess for the root |

**Memory trick:** `root(function, first_guess)`.

In [ ]:
# Break-even example: profit = 0
def profit(units):
    revenue = 50 * units                  # sell at 50 PKR each
    cost = 2000 + 30 * units              # fixed cost + cost per unit
    return revenue - cost                 # we want this to be zero

result = root(profit, 10)                 # 10 = first guess for units
print(result.x)                           # [100.]  -> sell 100 units to break even

### The page example: root of `x + cos(x)`

We want the `x` where `x + cos(x) = 0`. There is no easy formula, so we let SciPy search, starting from the guess `0`.

In [ ]:
from numpy import cos

def eqn(x):                    # the equation, written as a function
    return x + cos(x)

myroot = root(eqn, 0)          # 0 is the first guess
print(myroot.x)                # [-0.73908513], the root is stored in .x

**Check the answer yourself:** A root is correct when you put it back in the function and get (almost) **zero**. This is a very good habit.

In [ ]:
print(eqn(myroot.x))           # [0.], plugging the root back gives zero, so it is correct

## 4. How to read the result object

`root()` does not return only a number. It returns a **result object** with many details. The page shows `print(myroot)` to see all of it.

In [ ]:
print(myroot)

**The parts that matter (for beginners):**

| Part | Meaning | Value here |
|---|---|---|
| `x` | The root that was found | `[-0.73908513]` |
| `success` | Did the search work? | `True` |
| `message` | A short sentence about the result | The solution converged |
| `fun` | The function value at the root (should be about 0) | `[0.]` |
| `nfev` | How many times the function was called | 11 |
| `method` | The search method used (`hybr` is the default) | `hybr` |

**Important habit:** Always check `success` before you trust `x`. If `success` is `False`, the search failed, and `x` is not a real answer.

Note that `x` is a **NumPy array** with one value, `[-0.739...]`, not a plain number. To get the plain number, use `myroot.x[0]`.

In [ ]:
print(myroot.success)              # True, the search worked
print(myroot.x[0])                 # -0.7390851332151607, plain number
print(round(myroot.x[0], 4))       # -0.7391, rounded for a report

## 5. Why the starting guess `x0` matters

Some equations have **more than one root**. Which one you get depends on your first guess.

**Example:** `x**2 - 4 = 0` has two roots: `2` and `-2`.

**Real-life example:** There are two bus stops that are both 4 km from your house, one to the east and one to the west. If you start walking from the east side, you reach the east stop first.

In [ ]:
def g(x):
    return x**2 - 4

print(root(g, 1).x)       # [2.]   start on the positive side -> finds 2
print(root(g, -1).x)      # [-2.]  start on the negative side -> finds -2

**Lesson:** `x0` is not a small detail. A good first guess gives the answer you want, and a bad one may give a different root or fail.

## 6. Job 2: Finding the minimum with `minimize()`

**Short answer:** `minimize()` finds the `x` that makes the function **as small as possible**.

**The page's picture in words:** A function is like a curve with high points and low points.

- The highest point of all is the **global maximum**. The lowest point of all is the **global minimum**.
- Other small hills are **local maxima**. Other small valleys are **local minima**.

**Real-life example:** A road through mountains. The deepest valley in the whole country is the global minimum. A small dip near your village is a local minimum.

### The simple maths

At the very bottom of a valley, the ground is **flat**. The slope is zero. This is how the optimizer knows it has arrived.

Take the page's function: `f(x) = x**2 + x + 2`.

1. The slope formula (derivative) is `2x + 1`.
2. At the bottom, slope = 0, so `2x + 1 = 0`, which gives `x = -0.5`.
3. The lowest value is `f(-0.5) = 0.25 - 0.5 + 2 = 1.75`.

So we **know the answer by hand**: the minimum is at `x = -0.5` and the lowest value is `1.75`. Now let SciPy find it, and see whether it agrees.

**The arguments:**

| Argument | Meaning |
|---|---|
| `fun` | The function to minimize |
| `x0` | First guess |
| `method` | Which search method to use (default is `BFGS` when you give no bounds or constraints) |
| `callback` | A function called after every step |
| `options` | A dictionary of extra settings |

In [ ]:
def eqn(x):
    return x**2 + x + 2              # the page's function

mymin = minimize(eqn, 0, method='BFGS')    # start at 0, use the BFGS method
print(mymin)

### How to read the output

| Part | Meaning | Value |
|---|---|---|
| `x` | **Where** the minimum is | about `-0.5` |
| `fun` | **How low** the lowest value is | `1.75` |
| `success` | Did it work? | `True` |
| `jac` | The slope at the end. About 0 means we are at the bottom | about `0` |
| `nit` | Number of steps (iterations) | 2 |
| `nfev` | How many times the function was called | 8 |
| `hess_inv` | Advanced detail about the curve's bend. Ignore for now | - |

**Why is `x` shown as `-0.50000001` and not exactly `-0.5`?** The computer stops when it is "close enough" (this is called tolerance). A tiny difference like `0.00000001` is normal. Just round it.

**Remember:** `x` = **where**, `fun` = **how low**. Beginners often mix these up.

In [ ]:
print(round(mymin.x[0], 4))       # -0.5   where the minimum is
print(round(mymin.fun, 4))        # 1.75   the lowest value
print(mymin.success)              # True

## 7. See it: a picture of the minimum

A picture makes it clear. You know Matplotlib, so let us draw the curve and mark the point SciPy found.

In [ ]:
xs = np.linspace(-3, 2, 200)                       # 200 x values from -3 to 2
plt.plot(xs, eqn(xs), label='f(x) = x^2 + x + 2')  # draw the curve
plt.scatter(mymin.x, mymin.fun, color='red', zorder=3, label='minimum found by SciPy')
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title('The minimum is where the curve is lowest')
plt.legend()
plt.show()

## 8. The `method` argument

A method is the **strategy** for walking downhill. Different methods suit different problems. The page lists these legal values:

| Method | Simple meaning | Good to know |
|---|---|---|
| `'CG'` | Conjugate Gradient | Uses slopes, no extra setup |
| `'BFGS'` | A smart slope-based method | The default, a good first choice |
| `'Newton-CG'` | Uses slope and curve | You must also give the slope function (`jac`) |
| `'L-BFGS-B'` | A memory-saving version of BFGS | Allows `bounds` (limits on x). Good for big problems |
| `'TNC'` | Truncated Newton | Allows `bounds` |
| `'COBYLA'` | Does not use slopes | Allows constraints (rules like x must be above 0) |
| `'SLSQP'` | Sequential least squares | Allows bounds and constraints |

SciPy has more methods than these 7 (for example `Nelder-Mead`). The page just shows these.

**Beginner advice:** Do not memorize them. Start with the default (`BFGS`). Use `L-BFGS-B` or `SLSQP` later, when you need limits on `x`.

**Real-life example:** Different ways to find the lowest point in a city. You can walk downhill (slope methods), or you can test random streets without using a slope (COBYLA). All can work, but one is faster for a given city.

In [ ]:
slope = lambda x: 2 * x + 1               # slope of x**2 + x + 2 (needed only by Newton-CG)

for m in ['CG', 'BFGS', 'Newton-CG', 'L-BFGS-B', 'TNC', 'COBYLA', 'SLSQP']:
    j = slope if m == 'Newton-CG' else None           # only Newton-CG needs the slope function
    res = minimize(eqn, 0, method=m, jac=j)           # same problem, 7 different methods
    print(m, round(float(res.x[0]), 3))               # all find about -0.5

**Lesson:** All 7 methods find about the same answer for an easy problem. The difference appears in big, difficult problems.

## 9. Local minimum trap

For a curve with **several valleys**, the optimizer goes into the nearest valley from your starting point. It may stop in a small valley and never see the deepest one.

**Real-life example:** You walk downhill in fog and stop in a small village dip. The deepest valley is in the next mountain, but you cannot know that.

Example function: `f(x) = x**4 - 3*x**2 + x`. It has two valleys.

In [ ]:
def h(x):
    return x[0]**4 - 3 * x[0]**2 + x[0]      # x is an array, so we use x[0]

left = minimize(h, [-2])        # start on the left side
right = minimize(h, [2])        # start on the right side

print(round(left.x[0], 3), round(left.fun, 3))      # -1.301  -3.514  deeper valley (global)
print(round(right.x[0], 3), round(right.fun, 3))    #  1.131  -1.07   shallower valley (local)

**Lesson:** Starting from `2`, the optimizer found a valley with value `-1.07`. Starting from `-2`, it found a deeper valley with `-3.514`. Same function, different start, different answer.

**What to do about it:** Try **several starting points** and compare the `fun` values. Keep the lowest one.

**Note:** Here we wrote `x[0]` because `minimize()` always passes `x` to your function as an **array**, even when you gave it a single number as `x0`.

## 10. `options` and `callback`

**`options`** is a dictionary of extra settings. The page names two:

| Key | Meaning |
|---|---|
| `"disp"` | `True` prints a detailed description of the search |
| `"gtol"` | How small the slope must be before the search stops (a smaller number means a more exact answer) |

**`callback`** is a function that SciPy calls **after every step**. You can use it to watch the search, like a live tracker.

In [ ]:
steps = []                                         # empty list to collect each step

def watch(xk):                                     # called after every step with the current x
    steps.append(float(xk[0]))

minimize(eqn, 5, method='BFGS', callback=watch)    # start far away at 5
print(steps)    # [3.99, 1.72, -0.5]  -> 5 moves closer and closer to -0.5

In [ ]:
# options: print details and ask for a more exact stop
res = minimize(eqn, 0, method='BFGS', options={'disp': True, 'gtol': 1e-8})

## 11. Trick: how to find a maximum

SciPy has `minimize()` but **no** `maximize()`. The trick is simple: **to find the highest point of `f`, minimize `-f`**. Flipping the sign turns hills into valleys.

**Real-life example:** If you want to make the biggest profit, minimize the "negative profit", which is the same thing.

Next section uses this trick.

## 12. Real-life example: best advertising spend

A freelance client runs ads. From past data, profit (in thousands of PKR) depends on the ad spend (in thousands of PKR) like this:

`profit = -(spend - 40)**2 + 500`

Too little spend gives few customers. Too much spend wastes money. There is a best amount in the middle.

**Question:** What spend gives the maximum profit?

**By hand:** The top of this curve is at `spend = 40` with `profit = 500`. Let us see if SciPy finds it.

In [ ]:
def profit_fn(spend):
    return -(spend[0] - 40)**2 + 500          # profit curve, highest at spend = 40

best = minimize(lambda s: -profit_fn(s), [10])    # minus sign: minimize the negative to maximize

print(round(best.x[0], 2))                    # 40.0   best ad spend
print(round(-best.fun, 2))                    # 500.0  maximum profit (flip the sign back)

**Reading it:** `best.x` is the best spend, and `-best.fun` is the maximum profit. We flip the sign back because we minimized the negative.

## 13. Link to Machine Learning: fit a line by minimizing error

This is the most important section for your goal. Linear regression is just an optimizer problem.

**The idea:** A line is `sales = slope x ad_spend + intercept`. For any slope and intercept, we can measure how wrong the line is: take each point's gap from the line, **square** it, and add all gaps (this is called the sum of squared errors). The best line is the one with the **smallest** total error.

So: **training a model = minimizing the error.**

**Real-life example:** Ad spend (x) and sales (y) for 5 days (fake numbers).

- Our guess to check: the points follow roughly `y = 2x`, so slope should be near 2.

In [ ]:
x = np.array([1, 2, 3, 4, 5.])                # ad spend
y = np.array([2.1, 4.2, 5.9, 8.1, 10.2])      # sales

def sum_squared_error(p):                     # p[0] = slope, p[1] = intercept
    prediction = p[0] * x + p[1]              # the line's guess for every point (NumPy, no loop)
    return np.sum((y - prediction) ** 2)      # total squared mistake

fit = minimize(sum_squared_error, [0, 0])     # start with slope 0 and intercept 0
print(np.round(fit.x, 2))                     # [2.01 0.07]  slope and intercept found

**Check with a tool you know:** NumPy's `polyfit` solves the same problem directly. If both agree, we are right.

In [ ]:
print(np.round(np.polyfit(x, y, 1), 2))       # [2.01 0.07]  the same answer

**This is the big takeaway.** When you later use `LinearRegression()` in scikit-learn, it is doing this same kind of search for the line with the smallest error. Logistic regression does the same with a different error formula. Nothing magical, just minimizing.

**Hotel project link:** When you build your cancellation predictor, the model will be trained by minimizing an error (called log-loss) on your booking data.

## 14. Common mistakes

1. **Forgetting `x0`.** `root()` and `minimize()` both need a first guess.
2. **Trusting `x` without checking `success`.** If `success` is `False`, the answer is not reliable.
3. **Mixing up `x` and `fun`.** `x` = where, `fun` = the function value there.
4. **Thinking `x` is a plain number.** It is a NumPy array. Use `result.x[0]` for the number.
5. **Expecting `-0.5` exactly.** You will get `-0.50000001`. That is normal, so round it.
6. **Thinking the answer is always the global minimum.** It may be a local one. Try different starting points.
7. **Looking for `maximize()`.** It does not exist. Minimize the negative.
8. **Using `Newton-CG` without `jac`.** That method needs the slope function.

## 15. Cheat sheet (revise in 2 minutes)

- Optimizer = finds the best answer by repeated guessing. Two jobs: **root** (function = 0) and **minimum** (function lowest).
- `root(fun, x0)` returns an object. The answer is in `.x`. Check `.success`. Verify with `fun(result.x)`, which should be about 0.
- `minimize(fun, x0, method=...)` returns an object. `.x` = where, `.fun` = how low, `.jac` = slope at the end (about 0).
- `x` is an **array**, so use `x[0]` inside your function when you write `x[0]`.
- Default method is `BFGS`. Others: `CG`, `Newton-CG`, `L-BFGS-B`, `TNC`, `COBYLA`, `SLSQP`.
- `options={'disp': True, 'gtol': ...}` for detail and exactness. `callback=function` to watch every step.
- **Maximize = minimize the negative.**
- The starting guess matters, especially with several roots or valleys.
- **ML link:** training a model = minimizing its error.

## 16. Interview questions (try to answer first, then read)

**Q1. What is an optimizer?**
A tool that searches for the best value of a function, either where it equals zero (root) or where it is lowest (minimum).

**Q2. When do you use `root()` and when `minimize()`?**
Use `root()` to solve an equation like `f(x) = 0`, for example a break-even point. Use `minimize()` to find the lowest value of a function, for example the smallest error.

**Q3. Why can NumPy not solve `x + cos(x) = 0`?**
Because it is a non-linear equation with no simple formula. SciPy searches for the answer step by step.

**Q4. What are the two required arguments of `root()` and `minimize()`?**
`fun` (the function) and `x0` (the first guess).

**Q5. What is the difference between a global and a local minimum?**
The global minimum is the lowest point of the whole function. A local minimum is only the lowest point nearby.

**Q6. How do you find a maximum with SciPy?**
Minimize the negative of the function.

**Q7. Why is `x0` important?**
The search starts there. With several roots or valleys, different starting points can give different answers.

**Q8. How does optimization connect to Machine Learning?**
Training a model means changing its numbers to make the error as small as possible. That is a minimization problem.

**Q9. What does `jac` show in the output?**
The slope at the final point. A value near 0 means we reached a flat bottom.

## 17. Practice exercises (15 minutes)

1. **Root:** Find the root of `x**3 - 8` (answer should be about 2). Verify by plugging it back in.
2. **Two roots:** Find both roots of `x**2 - 9` by using two different `x0` values.
3. **Break-even:** A product sells for 80 PKR, costs 50 PKR per unit, and the fixed cost is 3000 PKR. How many units give zero profit? Check by hand first.
4. **Minimize:** Find the minimum of `(x - 3)**2 + 5`. Say what `x` and `fun` should be before you run it.
5. **Maximize:** Find the maximum of `-(x - 7)**2 + 20`. Use the minus-sign trick.
6. **Local trap:** Run the section 9 function from 4 different starting points and compare the `fun` values.

In [ ]:
# Space for my practice code

## 18. My own notes and review plan

**Write the main idea in 3 lines, in your own words (no copying):**

1.
2.
3.

**Review plan (spaced repetition):**

| Review | Date | Done? |
|---|---|---|
| Explain without looking | 28 Nov 2026 | |
| Explain again and redo exercises | 2 Dec 2026 | |

**Save to GitHub:**

```bash
git add 01_scipy/scipy_optimizers_notes.ipynb
git commit -m "Add SciPy optimizers notes"
git push
```

**Next topic:** SciPy Sparse Data, then SciPy Significance Tests (`scipy.stats`), the most important SciPy part for your data analysis and ML interviews.